In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, KFold, ShuffleSplit
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, cross_validate
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('housing.csv')
df = df.drop('ocean_proximity', axis=1)
df = df.drop('total_bedrooms', axis=1)

print(df.head())
print(df.info())

In [ ]:
X = df.drop('median_house_value', axis=1) 
y = df['median_house_value']               

print(f"Матрица признаков X: {X.shape}")
print(f"Признаки: {list(X.columns)}")
print(f"Вектор целевой переменной y: {y.shape}")


In [ ]:
scaler = StandardScaler()
x_scaled = scaler.fit_transform(X) 

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    x_scaled, y, test_size=0.2, random_state=1
)

print(f"Обучающая выборка: {x_train.shape[0]}")
print(f"Тестовая выборка: {x_test.shape[0]}")

In [ ]:
k_initial = 1

knn_initial = KNeighborsRegressor(n_neighbors=k_initial)
knn_initial.fit(x_train, y_train)

y_pred_initial = knn_initial.predict(x_test)

mae_initial = mean_absolute_error(y_test, y_pred_initial)
mse_initial = mean_squared_error(y_test, y_pred_initial)
rmse_initial = np.sqrt(mse_initial)
r2_initial = r2_score(y_test, y_pred_initial)

print(f"Оценка модели при K={k_initial}")
print(f"средняя абсолютная ошибка: {mae_initial:.4f}")
print(f"средняя квадратичная ошибка: {mse_initial:.4f}")
print(f"корень из средней квадратичной ошибки: {rmse_initial:.4f}")
print(f"коэффициент детерминации: {r2_initial:.4f}")


In [ ]:
param_grid = {'n_neighbors': range(1, 31)}

kf = KFold(n_splits=5, shuffle=True, random_state=1)
grid_search_kf = GridSearchCV(
    KNeighborsRegressor(), 
    param_grid, 
    cv=kf, 
    scoring='r2',
    n_jobs=-1
)
grid_search_kf.fit(x_train, y_train)

In [ ]:
ss = ShuffleSplit(n_splits=5, test_size=0.2, random_state=1)
grid_search_ss = GridSearchCV(
    KNeighborsRegressor(), 
    param_grid, 
    cv=ss, 
    scoring='r2',
    n_jobs=-1
)
grid_search_ss.fit(x_train, y_train)

In [ ]:
random_search = RandomizedSearchCV(
    KNeighborsRegressor(), 
    param_grid, 
    n_iter=15,
    cv=5, 
    scoring='r2',
    random_state=1,
    n_jobs=-1
)
random_search.fit(x_train, y_train)



In [ ]:
print(f"Лучшее K (KFold, 5 фолдов): {grid_search_kf.best_params_['n_neighbors']}")
print(f"Лучшее R² (KFold, 5 фолдов): {grid_search_kf.best_score_:.4f}\n\n")
print(f"Лучшее K (ShuffleSplit, 5 разбиений): {grid_search_ss.best_params_['n_neighbors']}")
print(f"Лучшее R² (ShuffleSplit, 5 разбиений): {grid_search_ss.best_score_:.4f}\n\n")
print(f"\nЛучшее K (RandomizedSearchCV): {random_search.best_params_['n_neighbors']}")
print(f"Лучшее R² (RandomizedSearchCV): {random_search.best_score_:.4f}")

In [ ]:
k_best = grid_search_kf.best_params_['n_neighbors']
knn_best = KNeighborsRegressor(n_neighbors=k_best)
knn_best.fit(x_train, y_train)



In [ ]:
y_pred_best = knn_best.predict(x_test)

mae_best = mean_absolute_error(y_test, y_pred_best)
mse_best = mean_squared_error(y_test, y_pred_best)
rmse_best = np.sqrt(mse_best)
r2_best = r2_score(y_test, y_pred_best)

print(f"Оценка оптимальной модели с K={k_best}")
print(f"средняя абсолютная ошибка: {mae_best:.4f}")
print(f"средняя квадратичная ошибка: {mse_best:.4f}")
print(f"корень из средней квадратичной ошибки: {rmse_best:.4f}")
print(f"коэффициент детерминации: {r2_best:.4f}\n\n")

print("Сравнение")
print(f"{'Метрика'} {'Исходная (K=' + str(k_initial) + ')'} {'Оптимальная (K=' + str(k_best) + ')'}")
print(f"{'средняя абсолютная ошибка'} {mae_initial:.4f} {mae_best:.4f}")
print(f"{'средняя квадратичная ошибка'} {mse_initial:.4f} {mse_best:.4f}")
print(f"{'корень из средней квадратичной ошибки'} {rmse_initial:.4f} {rmse_best:.4f}")
print(f"{'коэффициент детерминации'} {r2_initial:.4f} {r2_best:.4f}")

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(y_test, y_pred_initial, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Реальные значения Rating')
plt.ylabel('Предсказанные значения')
plt.title(f'Исходная модель (K={k_initial})\nR² = {r2_initial:.4f}')

plt.subplot(1, 2, 2)
plt.scatter(y_test, y_pred_best, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Реальные значения Rating')
plt.ylabel('Предсказанные значения')
plt.title(f'Оптимальная модель (K={k_best})\nR² = {r2_best:.4f}')

plt.tight_layout()
plt.show()

In [ ]:
k_range = range(1, 31)
cv_scores = []

for k in k_range:
    knn = KNeighborsRegressor(n_neighbors=k)
    scores = cross_val_score(knn, x_train, y_train, cv=5, scoring='r2')
    cv_scores.append(scores.mean())

plt.figure(figsize=(10, 6))
plt.plot(k_range, cv_scores, 'bo-')
plt.xlabel('Количество соседей (K)')
plt.ylabel('Средний R² (кросс-валидация)')
plt.title('Зависимость качества модели от гиперпараметра K')
plt.grid(True)
plt.show()

ЛР4

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.tree import plot_tree, export_text

In [ ]:
linear = LinearRegression()
linear.fit(x_train, y_train)
y_pred_linear = linear.predict(x_test)
print("Коэффициенты линейной регрессии:")
print(f"Свободный член (intercept): {linear.intercept_:.2f}")
print("Коэффициенты при признаках:")
for feature, coef in zip(X.columns, linear.coef_):
    print(f"  {feature:20s}: {coef:.4f}")

mae_linear = mean_absolute_error(y_test, y_pred_linear)
mse_linear = mean_squared_error(y_test, y_pred_linear)
rmse_linear = np.sqrt(mse_linear)
r2_linear = r2_score(y_test, y_pred_linear)

print("Линейная регрессия")
print(f"Средняя абсолютная ошибка:      {mae_linear:.2f}")
print(f"Средняя квадратичная ошибка:    {mse_linear:.2f}")
print(f"Корень из средней квадратичной ошибки:{rmse_linear:.2f}")
print(f"Коэффициент детерминации:       {r2_linear:.4f}")

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_linear, alpha=0.5, color='blue', edgecolors='k')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Реальные значения (median_house_value)')
plt.ylabel('Предсказанные значения')
plt.title(f'Линейная регрессия\nR² = {r2_linear:.4f}, RMSE = {rmse_linear:.2f}')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
param_grid_svr = {
    'C': [0.1, 1.0, 10.0, 100.0],     
    'epsilon': [0.01, 0.1, 0.5],        
    'gamma': ['scale', 'auto', 0.01, 0.1, 1.0] 
}

svr = GridSearchCV(
    SVR(kernel='rbf'),
    param_grid_svr,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

svr.fit(x_train, y_train)

In [ ]:
svr = svr.best_estimator_
y_pred_svr = svr.predict(x_test)

mae_svr = mean_absolute_error(y_test, y_pred_svr)
mse_svr = mean_squared_error(y_test, y_pred_svr)
rmse_svr = np.sqrt(mse_svr)
r2_svr = r2_score(y_test, y_pred_svr)

print("SVR")
print(f"Средняя абсолютная ошибка:              {mae_svr:.2f}")
print(f"Средняя квадратичная ошибка:            {mse_svr:.2f}")
print(f"Корень из средней квадратичной ошибки:  {rmse_svr:.2f}")
print(f"Коэффициент детерминации:               {r2_svr:.4f}")

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_svr, alpha=0.5, color='green', edgecolors='k')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Реальные значения (median_house_value)')
plt.ylabel('Предсказанные значения')
plt.title(f'SVR (оптимальная модель)\nC={svr.C}, epsilon={svr.epsilon}, gamma={svr.gamma}\nR² = {r2_svr:.4f}, RMSE = {rmse_svr:.2f}')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
param_grid_dt = {
    'max_depth': [3, 5, 7, 10, 15, None],        
    'min_samples_split': [2, 5, 10, 20],         
    'min_samples_leaf': [1, 2, 5, 10],           
    'max_features': ['sqrt', 'log2', None]      
}

dt = GridSearchCV(
    DecisionTreeRegressor(random_state=1),
    param_grid_dt,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

dt.fit(x_train, y_train)


In [ ]:
for param, value in dt.best_params_.items():
    print(f"  {param}: {value}")

dt = dt.best_estimator_
y_pred_dt = dt.predict(x_test)

mae_dt = mean_absolute_error(y_test, y_pred_dt)
mse_dt = mean_squared_error(y_test, y_pred_dt)
rmse_dt = np.sqrt(mse_dt)
r2_dt = r2_score(y_test, y_pred_dt)

print("Дерево решений")
print(f"Средняя абсолютная ошибка:              {mae_dt:.2f}")
print(f"Средняя квадратичная ошибка:            {mse_dt:.2f}")
print(f"Корень из средней квадратичной ошибки:  {rmse_dt:.2f}")
print(f"Коэффициент детерминации:               {r2_dt:.4f}")
print(f"  Количество листьев: {dt.get_n_leaves()}")
print(f"  Глубина дерева: {dt.get_depth()}")

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_dt, alpha=0.5, color='orange', edgecolors='k')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Реальные значения (median_house_value)')
plt.ylabel('Предсказанные значения')
plt.title(f'Дерево решений (оптимальная модель)\nmax_depth={dt.max_depth}, min_samples_split={dt.min_samples_split}\nR² = {r2_dt:.4f}, RMSE = {rmse_dt:.2f}')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(20, 12))
plot_tree(dt, 
          feature_names=list(X.columns),
          filled=True, 
          rounded=True,
          fontsize=10,
          max_depth=3,
          precision=2)
plt.title('Дерево решений (первые 3 уровня)')
plt.tight_layout()
plt.show()

In [ ]:
tree_rules = export_text(dt, feature_names=list(X.columns), max_depth=4)
print(tree_rules)

In [ ]:
feature_importance = dt.feature_importances_

sorted_idx = np.argsort(feature_importance)[::-1]

print("Ранжирование признаков по важности:")
for i, idx in enumerate(sorted_idx):
    print(f"{i+1}. {X.columns[idx]:25s}: {feature_importance[idx]:.4f} ({feature_importance[idx]*100:.2f}%)")

plt.figure(figsize=(10, 8))
colors = plt.cm.viridis(np.linspace(0, 1, len(X.columns)))
bars = plt.barh(range(len(X.columns)), feature_importance[sorted_idx], color=colors[sorted_idx])
plt.yticks(range(len(X.columns)), [X.columns[i] for i in sorted_idx])
plt.xlabel('Важность признака (Feature Importance)')
plt.ylabel('Признаки')
plt.title('Важность признаков в дереве решений\n(Чем больше значение, тем важнее признак для предсказания)')
plt.gca().invert_yaxis() 
plt.grid(axis='x', alpha=0.3)

for i, (bar, importance) in enumerate(zip(bars, feature_importance[sorted_idx])):
    plt.text(importance + 0.005, bar.get_y() + bar.get_height()/2, 
             f'{importance:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
results = {
    'Модель': ['Линейная регрессия', 'SVR', 'Дерево решений'],
    'MAE': [mae_linear, mae_svr, mae_dt],
    'RMSE': [rmse_linear, rmse_svr, rmse_dt],
    'R²': [r2_linear, r2_svr, r2_dt]
}

results_df = pd.DataFrame(results)
results_df = results_df.round({'MAE': 2, 'RMSE': 2, 'R²': 4})

print(results_df.to_string(index=False))

best_mae_idx = np.argmin([mae_linear, mae_svr, mae_dt])
best_rmse_idx = np.argmin([rmse_linear, rmse_svr, rmse_dt])
best_r2_idx = np.argmax([r2_linear, r2_svr, r2_dt])

models_list = ['Линейная регрессия', 'SVR', 'Дерево решений']

print(f"  Лучшая средняя абсолютная ошибка:  {models_list[best_mae_idx]} ({min(mae_linear, mae_svr, mae_dt):.2f})")
print(f"  Лучший корень из средней квадратичной ошибки: {models_list[best_rmse_idx]} ({min(rmse_linear, rmse_svr, rmse_dt):.2f})")
print(f"  Лучший коэффициент детерминации:   {models_list[best_r2_idx]} ({max(r2_linear, r2_svr, r2_dt):.4f})")


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].bar(results_df['Модель'], results_df['MAE'], color=['blue', 'green', 'orange'])
axes[0].set_ylabel('MAE (доллары)')
axes[0].set_title('Сравнение MAE (чем меньше, тем лучше)')
axes[0].grid(axis='y', alpha=0.3)

for i, (model, mae) in enumerate(zip(results_df['Модель'], results_df['MAE'])):
    axes[0].text(i, mae + 500, f'{mae:.0f}', ha='center', fontsize=10)

axes[1].bar(results_df['Модель'], results_df['RMSE'], color=['blue', 'green', 'orange'])
axes[1].set_ylabel('RMSE (доллары)')
axes[1].set_title('Сравнение RMSE (чем меньше, тем лучше)')
axes[1].grid(axis='y', alpha=0.3)

for i, (model, rmse) in enumerate(zip(results_df['Модель'], results_df['RMSE'])):
    axes[1].text(i, rmse + 500, f'{rmse:.0f}', ha='center', fontsize=10)

axes[2].bar(results_df['Модель'], results_df['R²'], color=['blue', 'green', 'orange'])
axes[2].set_ylabel('R² (коэффициент детерминации)')
axes[2].set_title('Сравнение R² (чем ближе к 1, тем лучше)')
axes[2].set_ylim(0, 1)
axes[2].grid(axis='y', alpha=0.3)

for i, (model, r2) in enumerate(zip(results_df['Модель'], results_df['R²'])):
    axes[2].text(i, r2 + 0.02, f'{r2:.4f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()